In [144]:
pip install transformers

In [145]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

In [146]:
dataset=load_dataset("csv",data_files="/content/sentiment_analysis.csv")

In [147]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Year', 'Month', 'Day', 'Time of Tweet', 'text', 'sentiment', 'Platform'],
        num_rows: 499
    })
})

In [148]:
dataset["train"]

Dataset({
    features: ['Year', 'Month', 'Day', 'Time of Tweet', 'text', 'sentiment', 'Platform'],
    num_rows: 499
})

In [149]:
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

In [150]:
def tokenizer_function(example):
  return tokenizer(example["text"], padding="max_length", truncation=True, max_length=512)

In [151]:
dataset=dataset.map(tokenizer_function)

In [152]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Year', 'Month', 'Day', 'Time of Tweet', 'text', 'sentiment', 'Platform', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 499
    })
})

In [153]:
dataset=dataset.rename_column("sentiment","labels")

In [154]:
dataset=dataset.remove_columns([
    "Year","Month","Day","Time of Tweet","text"
])

In [155]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'Platform', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 499
    })
})

In [156]:
dataset=dataset.remove_columns(["Platform"])

In [157]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 499
    })
})

### Map String Labels to Integer IDs

The `BertForSequenceClassification` model expects numerical labels (e.g., 0, 1, 2) rather than string labels (e.g., 'positive', 'negative'). We need to create a mapping from our string sentiments to unique integer IDs and apply this to the dataset.

In [158]:
unique_labels = sorted(list(set(dataset["train"]["labels"])))
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for i, label in enumerate(unique_labels)}

print(f"Unique labels found: {unique_labels}")
print(f"Label to ID mapping: {label_to_id}")
print(f"ID to Label mapping: {id_to_label}")

Unique labels found: ['negative', 'neutral', 'positive']
Label to ID mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
ID to Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [159]:
def map_labels_to_ids(example):
    example["labels"] = label_to_id[example["labels"]]
    return example

dataset = dataset.map(map_labels_to_ids)

# Verify the change in labels
print("First 5 labels after mapping:")
print(dataset["train"]["labels"][:5])

# Check the feature type for 'labels'
print("\nFeatures after mapping:")
print(dataset["train"].features)

First 5 labels after mapping:
[2, 2, 0, 0, 0]

Features after mapping:
{'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


In [160]:
dataset["train"].set_format("torch", columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [161]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 499
    })
})

In [162]:
dataset["train"]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 499
})

In [163]:
from transformers import Trainer

In [164]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

In [165]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [166]:
model=BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [167]:
from transformers import TrainingArguments

In [169]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="steps",   # 🔥 important
    logging_steps=10,           # log every 10 steps
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=15,
    learning_rate=2e-5,
    fp16=True
)

In [170]:
import torch

In [171]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [172]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [173]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    compute_metrics=compute_metrics
)

In [174]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.115188,1.072149,0.370000
2,1.027651,1.014106,0.530000
3,0.882275,0.877300,0.640000
4,0.708763,0.764440,0.700000
5,0.469109,0.643250,0.750000
6,0.287455,0.637874,0.740000
7,0.186400,0.576515,0.810000
8,0.081430,0.625652,0.810000
9,0.040078,0.639873,0.810000
10,0.030028,0.717437,0.810000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=255, training_loss=0.32377602666908617, metrics={'train_runtime': 171.6006, 'train_samples_per_second': 34.878, 'train_steps_per_second': 1.486, 'total_flos': 1574733805102080.0, 'train_loss': 0.32377602666908617, 'epoch': 15.0})

In [175]:
trainer.evaluate()

{'eval_loss': 0.7897172570228577,
 'eval_accuracy': 0.81,
 'eval_runtime': 0.8106,
 'eval_samples_per_second': 123.367,
 'eval_steps_per_second': 6.168,
 'epoch': 15.0}

In [176]:
from transformers import BertTokenizer

In [182]:
text="i love you"

In [180]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')


In [183]:
text=tokenizer_function(text)

TypeError: string indices must be integers, not 'str'

In [179]:
text

{'input_ids': [101, 1045, 2293, 2017, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [185]:
import torch

text = "I love this product, it's amazing!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)
inputs.to(device)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits, dim=1).item()

print("Prediction:", pred)

Prediction: 2


In [186]:
text2="i hate you"
text2=tokenizer(
    text2,
    return_tensors="pt",
    truncation=True,
    padding=True
)
text2.to(device)
with torch.no_grad():
  output=model(**text2)

print(output)

SequenceClassifierOutput(loss=None, logits=tensor([[ 3.3594, -1.6670, -1.5732]], device='cuda:0'), hidden_states=None, attentions=None)


In [189]:
pred=output.logits

In [190]:
torch.argmax(pred,dim=1).item()

0

In [ ]:
model.save("sentiment_model.")

In [191]:
trainer.save_model("./my_model")
tokenizer.save_pretrained("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model/tokenizer_config.json', './my_model/tokenizer.json')

In [192]:
tokenizer=BertTokenizer.from_pretrained("./my_model")

In [193]:
model=BertForSequenceClassification.from_pretrained("./my_model")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [195]:
text="fuck you"
text=tokenizer(text,padding=True,return_tensors="pt")
print(text)

{'input_ids': tensor([[ 101, 6616, 2017,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [196]:
with torch.no_grad():
  output=model(**text)

In [197]:
print(output)

SequenceClassifierOutput(loss=None, logits=tensor([[ 3.1869, -1.1973, -2.0739]]), hidden_states=None, attentions=None)


In [198]:
output.logits

tensor([[ 3.1869, -1.1973, -2.0739]])

In [199]:
torch.argmax(output.logits)

tensor(0)

In [200]:
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model("/content/drive/MyDrive/my_model")
tokenizer.save_pretrained("/content/drive/MyDrive/my_model")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/my_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_model/tokenizer.json')